# Example: Running an Adaptive Rebalancing Engine
In this example, we place a signal-dependent allocation rule inside a guarded rebalancing loop and compare it with a static portfolio on the same synthetic market path.

> __Learning Objectives:__
>
> By the end of this example, you will be able to:
>
> * __Generate adaptive target weights:__ Map a market-state signal into defensive or aggressive preferences.
> * __Apply explicit triggers:__ Rebalance only when the schedule, allocation drift, or drawdown condition fires.
> * __Evaluate an engine scorecard:__ Compare wealth, drawdown, turnover, costs, and intervention counts against a static baseline.

Let's test whether adaptation adds value after accounting for its operational burden.
___


## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading the packages used in this example.

> __Include:__ The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates `Include.jl` in the notebook's global scope. The file activates the course environment, defines notebook-relative paths, and loads the required packages.

Let's set up the code environment:

The reusable portfolio algorithms in this example are provided by the local [`VLQuantitativeFinancePackage.jl`](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/) package.


In [ ]:
include(joinpath(@__DIR__, "Include.jl"));


For additional information, see the [Julia documentation](https://docs.julialang.org/en/v1/) and the [CHEME 5660 documentation](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/).

___


## Task 1: Generate a Two-Regime Market Path
We simulate four assets. The first half has positive market drift and moderate correlation; the second half has negative drift, higher volatility, and stronger correlation. The path is shared by both strategies.


In [ ]:
Random.seed!(5660);
tickers = ["ALFA", "BRAV", "CHAR", "DELT"];
T = 504;
N = length(tickers);
β = [0.65, 0.95, 1.25, 1.55];
idiosyncratic_vol = [0.10, 0.12, 0.16, 0.20]/sqrt(252);

market_returns = vcat(
    rand(Normal(0.10/252, 0.14/sqrt(252)), T÷2),
    rand(Normal(-0.08/252, 0.28/sqrt(252)), T÷2),
);
asset_returns = Matrix{Float64}(undef, T, N);
for t in 1:T, i in 1:N
    asset_returns[t, i] = 0.01/252 + β[i]*market_returns[t] +
        idiosyncratic_vol[i]*randn();
end
prices = 100 .* exp.(cumsum(asset_returns, dims=1));
market_index = 100 .* exp.(cumsum(market_returns));


## Task 2: Define the Signal, Target, and Trigger Rules
The market-state signal is a bounded transformation of short- versus long-horizon exponential moving averages. A positive signal increases exposure to high-$\beta$ assets; a negative signal favors defensive assets.

The engine rebalances when at least one condition holds: 21 trading days have elapsed, maximum allocation drift exceeds 8%, or portfolio drawdown exceeds 10%.


In [ ]:
short_ema = compute_ema(market_index, 10);
long_ema = compute_ema(market_index, 42);
signal = tanh.(8 .* (short_ema ./ long_ema .- 1));


## Task 3: Compare Adaptive and Static Policies
Both policies see the same returns. The static strategy keeps its initial allocation, while the adaptive strategy pays transaction costs whenever a trigger fires.


In [ ]:
adaptive_result = run_rebalancing_engine(asset_returns, signal, β; adaptive=true);
static_result = run_rebalancing_engine(asset_returns, signal, β; adaptive=false);

scorecard = DataFrame(
    policy=["Adaptive", "Static"],
    terminal_wealth=[adaptive_result.wealth[end], static_result.wealth[end]],
    max_drawdown=[adaptive_result.max_drawdown, static_result.max_drawdown],
    cumulative_turnover=[adaptive_result.turnover, static_result.turnover],
    transaction_cost=[adaptive_result.costs, static_result.costs],
    interventions=[adaptive_result.interventions, static_result.interventions],
);
pretty_table(scorecard; table_format=TextTableFormat(borders=text_table_borders__simple))


In [ ]:
plot(0:T, adaptive_result.wealth, lw=2, c=:navy,
    label="Adaptive engine", xlabel="Trading day", ylabel="Wealth")
plot!(0:T, static_result.wealth, lw=2, c=:gray, ls=:dash, label="Static allocation")
vline!([T÷2], c=:red, ls=:dot, label="Regime change")


## Summary
This example evaluated adaptation against the costs and interventions required to produce it.

> __Key Takeaways:__
>
> * __A signal is not yet a strategy:__ It must be connected to target weights, triggers, limits, and execution costs.
> * __Adaptive policies need a declared baseline:__ Performance should be compared on the same realized paths.
> * __Operational metrics matter:__ Turnover, costs, and intervention counts help explain why wealth and drawdown changed.

The engine is a candidate for validation, not evidence of deployment readiness.
___

## Disclaimer and Risks
This material is for educational purposes only and does not constitute investment advice. Results depend on the synthetic path, signal design, and simplified execution assumptions.
